<a href="https://colab.research.google.com/github/AdiY2j/CS6910_Assignment3/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
SOS_char = 0
EOS_char = 1

In [4]:
class Lang:
  def __init__(self, name):
    self.name = name
    self.word2count = {}
    self.word2index = {}
    self.index2word = {SOS_char : '<', EOS_char : '>'}
    self.n_chars = 2

  def add_word(self, word):
    for c in word:
      self.add_char(c)

  def add_char(self, char):
    if char not in self.word2index: # If char not present add it in word2index and inc counter
      self.word2index[char] = self.n_chars
      self.word2count[char] = 1
      self.index2word[self.n_chars] = char
      self.n_chars += 1
    else:
      self.word2count[char] += 1 #If char already present just increment counter

In [5]:
train_data = pd.read_csv('/content/drive/MyDrive/aksharantar_sampled/hin/hin_train.csv')
valid_data = pd.read_csv('/content/drive/MyDrive/aksharantar_sampled/hin/hin_valid.csv')

train_data = np.array(train_data)
valid_data = np.array(valid_data)

In [6]:
train_X, train_y = train_data[:,0], train_data[:,1]

In [7]:
input_lang, output_lang = Lang('eng'), Lang('hin')
for word in train_X:
  input_lang.add_word(word)
for word in train_y:
  output_lang.add_word(word)

pairs = [[train_X[i], train_y[i]] for i in range(len(train_X))]

In [8]:
print(pairs[1])
print(input_lang.n_chars, output_lang.n_chars)

['kirankant', 'किरणकांत']
28 66


In [9]:
len(output_lang.word2count)

64

In [10]:
input_word, output_word = pairs[1][0], pairs[1][1]
encoded_output = [output_lang.word2index[char] for char in output_word]
print(encoded_output)
decoded_output = [output_lang.index2word[i] for i in encoded_output]
print(decoded_output)

[9, 3, 10, 11, 9, 8, 12, 13]
['क', 'ि', 'र', 'ण', 'क', 'ा', 'ं', 'त']


In [11]:
def getIndex(lang, word):
  index = []
  for c in word:
    index.append(lang.word2index[c])

  return index

def getWordTensor(lang, word):
  index = getIndex(lang, word)
  index.append(EOS_char)
  return torch.tensor(index, dtype=torch.long, device=device).view(-1, 1)

def getTensorPairs(data):
  inputWordTensor = getWordTensor(input_lang, data[0])
  outputWordTensor = getWordTensor(output_lang, data[1])
  return (inputWordTensor, outputWordTensor)

In [103]:
class Encoder(nn.Module):
  def __init__(self, input_size, hidden_size, embedding_size, num_layers, dropout, cell_type, batch_size):
    super(Encoder, self).__init__()
    self.hidden_size = hidden_size
    self.embedding_size = embedding_size
    self.num_layers = num_layers
    self.batch_size = batch_size
    self.cell_type = cell_type
    self.embedding = nn.Embedding(input_size, embedding_size)
    self.dropout = nn.Dropout(dropout)

    match cell_type:
      case "RNN":
        self.rnn = nn.RNN(embedding_size, hidden_size, num_layers=num_layers, dropout=dropout)
      case "LSTM":
        self.rnn = nn.LSTM(embedding_size, hidden_size, num_layers=num_layers, dropout=dropout)
      case "GRU":
        self.rnn = nn.GRU(embedding_size, hidden_size, num_layers=num_layers, dropout=dropout)

  def initializeHidden(self):
    return torch.zeros(self.num_layers, 1, self.hidden_size, device=device)

  def forward(self, input, hidden, cell):
    embedded = self.embedding(input).view(1, 1, -1)
    if self.cell_type == "LSTM":
      output, (hidden, cell) = self.rnn(self.dropout(embedded), (hidden, cell))
    else:
      output, hidden = self.rnn(self.dropout(embedded), hidden)
    return output, hidden, cell